# Subject-level gender MAE-gap regularized residual multimodal Transformer

This experiment keeps the residual multimodal Transformer architecture, data
processing, subject split, optimizer, and evaluation protocol identical to the
completed baseline stability experiment. The only training change is a simple
gender fairness regularizer that first averages errors per subject and then
penalizes the difference between female and male mean subject errors.

Regularization strength is selected using validation predictions only. The
selected model is evaluated across five seeds and compared with the
completed unregularized residual Transformer baseline. Test predictions are
never used for checkpoint or hyperparameter selection.


In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from tqdm import tqdm
import json
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df.head()

,group,time,time_sec,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id
0,group01,2026-05-08 10:40:43.047895,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01
1,group01,2026-05-08 10:40:43.195317,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01
2,group01,2026-05-08 10:40:43.295405,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01
3,group01,2026-05-08 10:40:43.395835,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [3]:
df.shape

(889685, 68)

In [4]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH -1  # remove a sequence only when all the images are unavailable

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5       # 0=no weighting, 1=full inverse-frequency weighting
STRATIFY_COLUMN = 'gender'
BATCH_SIZE = 32
NUM_WORKERS = 8

ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]

In [5]:
df[['attention', 'image_path', 'age', 'gender']].isna().sum()

attention     0
image_path    0
age           0
gender        0
dtype: int64

In [6]:
sensor_cols = [
    col for col in df.columns
    if (
       # "sensor" in col.lower()
        "acceleration" in col.lower()
        or "gyro" in col.lower()
      #  or "rotation" in col.lower()
        or "heart" in col.lower()
      #  or "light" in col.lower()
        or "accel" in col.lower()
    )
    and "std" not in col.lower()
]

# Heart rate is physiologically different from motion/device sensors, so Fusion
# follows Temporal Sensor and gives it a separate projection stream.
hr_cols = ["heart_rate"] if "heart_rate" in sensor_cols else []
motion_cols = [col for col in sensor_cols if col not in hr_cols]
hr_indices = [sensor_cols.index(col) for col in hr_cols]
motion_indices = [sensor_cols.index(col) for col in motion_cols]

len(sensor_cols), len(motion_cols), len(hr_cols), sensor_cols[:5]

(9,
 8,
 1,
 ['lsm6dso_gyroscope value0_mean',
  'lsm6dso_gyroscope value1_mean',
  'lsm6dso_gyroscope value2_mean',
  'samsung_linear_acceleration_sensor value0_mean',
  'samsung_linear_acceleration_sensor value1_mean'])

In [7]:
# Treat physiologically impossible HR values as missing before missing flags and scaling.
# The smartwatch can emit 0, which should not be interpreted as a real heart rate.
df.loc[df["heart_rate"] < 30, "heart_rate"] = np.nan

df[sensor_cols] = df[sensor_cols].astype(np.float32)

In [8]:
# keep only 1 frame per second (the first)
df_sec = (df.sort_values(["subject_experiment_id", "time_sec"])
      .groupby(["subject_experiment_id", "time_sec"])
      .first()
      .reset_index())
df_sec.shape

(111754, 69)

In [9]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(
            sequence_df["time_sec"].min(),
            sequence_df["time_sec"].max() + 1
        )
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

sequence_metadata = (
    df[["subject_experiment_id", "subject_id", "gender", "age", "age_group"]]
    .drop_duplicates("subject_experiment_id"))

# Restore metadata for reconstructed missing seconds. Sensor/image/target values
# stay missing unless observed, so missingness flags remain meaningful.
temporal_frame_dataset = temporal_frame_dataset.drop(
    columns=["subject_id", "gender", "age", "age_group"],
    errors="ignore").merge(sequence_metadata, on="subject_experiment_id", how="left")

temporal_frame_dataset["visual_missing"] = temporal_frame_dataset["image_path"].isna().astype(np.float32)
temporal_frame_dataset["motion_missing"] = temporal_frame_dataset[motion_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["hr_missing"] = temporal_frame_dataset[hr_cols].isna().all(axis=1).astype(np.float32) if hr_cols else 1.0
temporal_frame_dataset["sensor_missing"] = temporal_frame_dataset[sensor_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["sensor_partial_nan"] = (
    temporal_frame_dataset[sensor_cols].isna().any(axis=1)
    & ~temporal_frame_dataset[sensor_cols].isna().all(axis=1)
).astype(np.float32)

temporal_frame_dataset[["subject_experiment_id", "time_sec", "visual_missing", "motion_missing", "hr_missing", "sensor_missing", "attention"]].head()

,subject_experiment_id,time_sec,visual_missing,motion_missing,hr_missing,sensor_missing,attention
0,group01_experiment01_subject_01,0,0.0,0.0,0.0,0.0,3.25
1,group01_experiment01_subject_01,1,0.0,0.0,0.0,0.0,3.00
2,group01_experiment01_subject_01,2,0.0,0.0,0.0,0.0,3.00
3,group01_experiment01_subject_01,3,0.0,0.0,0.0,0.0,3.25
4,group01_experiment01_subject_01,4,0.0,0.0,0.0,0.0,3.25


In [10]:
temporal_frame_dataset.shape

(121631, 74)

In [11]:
feature_index = pd.read_parquet('Data/clip_vitl14_features/clip_vitl14_frame_index.parquet')
feature_store_path =  'Data/clip_vitl14_features/clip_vitl14_frame_features.npy'
VISUAL_FEATURE_DIM = 768

feature_row_by_path = dict(zip(feature_index["image_path"], feature_index["feature_row"]))
FEATURES = feature_store_path.split('/')[-2]

temporal_frame_dataset["feature_row"] = (
    temporal_frame_dataset["image_path"]
    .map(feature_row_by_path)
    .fillna(-1)
    .astype(np.int64))

missing_feature_rows = (
    (temporal_frame_dataset["visual_missing"] == 0)
    & (temporal_frame_dataset["feature_row"] == -1)
).sum()
print("Non-missing image rows without cached features:", missing_feature_rows)

Non-missing image rows without cached features: 0


In [13]:
# Select one fixed age-stratified split using demographic metadata only.
# Model predictions, labels, and test performance are never used to choose it.
SPLIT_SEARCH_TRIALS = 1_000
MIN_MALE_VAL_SUBJECTS = 3
MIN_MALE_TEST_SUBJECTS = 3
MIN_AGE_GROUP_SUBJECTS_VAL = 1
MIN_AGE_GROUP_SUBJECTS_TEST = 1


def demographic_distance(split_df, full_df, column):
    categories = sorted(full_df[column].astype(str).unique())
    full_dist = full_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    split_dist = split_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    return float(np.abs(split_dist - full_dist).sum())


def find_constrained_age_stratified_split(subject_df, trials=SPLIT_SEARCH_TRIALS, seed=SEED):
    candidates = []
    all_age_groups = set(subject_df["age_group"].astype(str))

    for offset in range(trials):
        split_seed = seed + offset
        try:
            train_sub, temp_sub = train_test_split(
                subject_df,
                test_size=0.3,
                stratify=subject_df["age_group"],
                random_state=split_seed,
            )
            val_sub, test_sub = train_test_split(
                temp_sub,
                test_size=0.5,
                stratify=temp_sub["age_group"],
                random_state=split_seed,
            )
        except ValueError:
            continue

        val_males = int((val_sub["gender"].astype(str).str.lower() == "male").sum())
        test_males = int((test_sub["gender"].astype(str).str.lower() == "male").sum())
        val_min_age_count = int(
            val_sub["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
        )
        test_min_age_count = int(
            test_sub["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
        )

        representation_penalty = (
            max(0, MIN_MALE_VAL_SUBJECTS - val_males) * 100
            + max(0, MIN_MALE_TEST_SUBJECTS - test_males) * 100
            + max(0, MIN_AGE_GROUP_SUBJECTS_VAL - val_min_age_count) * 100
            + max(0, MIN_AGE_GROUP_SUBJECTS_TEST - test_min_age_count) * 100
        )
        balance_score = sum(
            demographic_distance(split, subject_df, column)
            for split in [train_sub, val_sub, test_sub]
            for column in ["age_group", "gender"]
        )
        candidates.append(
            (
                representation_penalty,
                balance_score,
                split_seed,
                train_sub.copy(),
                val_sub.copy(),
                test_sub.copy(),
            )
        )

    if not candidates:
        raise RuntimeError("Could not construct an age-stratified subject split.")

    best = min(candidates, key=lambda item: (item[0], item[1], item[2]))
    if best[0] > 0:
        raise RuntimeError(
            "No 70-15-15 split satisfied every requested representation constraint."
        )
    return best[3], best[4], best[5], best[2], best[0], best[1]


subject_df = (
    temporal_frame_dataset[["subject_id", "age_group", "gender"]]
    .dropna(subset=["subject_id", "age_group", "gender"])
    .drop_duplicates("subject_id")
    .copy()
)

train_sub, val_sub, test_sub, SPLIT_RANDOM_STATE, split_penalty, split_balance_score = (
    find_constrained_age_stratified_split(subject_df)
)

train_subjects = train_sub["subject_id"]
val_subjects = val_sub["subject_id"]
test_subjects = test_sub["subject_id"]

train_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(train_subjects)].copy()
val_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(val_subjects)].copy()
test_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(test_subjects)].copy()

# Train-only sensor normalization, identical to the original notebook.
sensor_means = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].mean()
sensor_stds = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].std()
sensor_stds = sensor_stds.replace(0, np.nan).fillna(1.0)
sensor_means = sensor_means.fillna(0.0)


def apply_sensor_scaling(frame_df):
    frame_df = frame_df.copy()
    scaled = ((frame_df[sensor_cols] - sensor_means) / sensor_stds).astype(np.float32)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled.loc[frame_df["sensor_missing"] == 1, :] = 0.0
    frame_df.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
    return frame_df


train_frames = apply_sensor_scaling(train_frames)
val_frames = apply_sensor_scaling(val_frames)
test_frames = apply_sensor_scaling(test_frames)

MOTION_INPUT_DIM = len(motion_cols) + 1
HR_INPUT_DIM = len(hr_cols) + 1
SENSOR_INPUT_DIM = len(sensor_cols) + 1


def split_demographic_table(split_df, split_name):
    rows = []
    for attribute in ["age_group", "gender"]:
        for group, count in split_df[attribute].astype(str).value_counts().sort_index().items():
            rows.append({
                "split": split_name,
                "attribute": attribute,
                "group": group,
                "subjects": int(count),
                "proportion": float(count / len(split_df)),
            })
    return pd.DataFrame(rows)


split_demographics = pd.concat(
    [
        split_demographic_table(train_sub, "train"),
        split_demographic_table(val_sub, "validation"),
        split_demographic_table(test_sub, "test"),
    ],
    ignore_index=True,
)

print(f"Selected demographic-only split random state: {SPLIT_RANDOM_STATE}")
print(f"Constraint penalty: {split_penalty}; demographic balance score: {split_balance_score:.4f}")
display(split_demographics)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_frames), len(val_frames), len(test_frames)],
    "subjects": [train_frames.subject_id.nunique(), val_frames.subject_id.nunique(), test_frames.subject_id.nunique()],
    "target_mean": [train_frames.attention.mean(), val_frames.attention.mean(), test_frames.attention.mean()],
    "visual_missing_rate": [train_frames.visual_missing.mean(), val_frames.visual_missing.mean(), test_frames.visual_missing.mean()],
    "motion_missing_rate": [train_frames.motion_missing.mean(), val_frames.motion_missing.mean(), test_frames.motion_missing.mean()],
    "hr_missing_rate": [train_frames.hr_missing.mean(), val_frames.hr_missing.mean(), test_frames.hr_missing.mean()],
    "sensor_missing_rate": [train_frames.sensor_missing.mean(), val_frames.sensor_missing.mean(), test_frames.sensor_missing.mean()],
})


Selected demographic-only split random state: 45
Constraint penalty: 0; demographic balance score: 0.3941


,split,attribute,group,subjects,proportion
0,train,age_group,"(13, 20]",13,0.333333
1,train,age_group,"(20, 22]",10,0.256410
2,train,age_group,"(22, 26]",10,0.256410
3,train,age_group,"(26, 44]",6,0.153846
4,train,gender,female,27,0.692308
5,train,gender,male,12,0.307692
6,validation,age_group,"(13, 20]",3,0.333333
7,validation,age_group,"(20, 22]",3,0.333333
8,validation,age_group,"(22, 26]",2,0.222222
9,validation,age_group,"(26, 44]",1,0.111111


,split,rows,subjects,target_mean,visual_missing_rate,motion_missing_rate,hr_missing_rate,sensor_missing_rate
0,train,84989,39,2.968285,0.086705,0.086705,0.095083,0.086705
1,val,19092,9,2.944667,0.070920,0.070920,0.074324,0.070920
2,test,17550,9,2.879391,0.065755,0.065755,0.114701,0.065755


In [14]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            visual_missing_flags = history["visual_missing"].astype(np.float32).values

            if visual_missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            motion_missing_flags = history["motion_missing"].astype(np.float32).values
            hr_missing_flags = history["hr_missing"].astype(np.float32).values
            sensor_missing_flags = history["sensor_missing"].astype(np.float32).values

            sequence_record = {
                "subject_experiment_id": sequence_id,
                "subject_id": sequence_df.iloc[i]["subject_id"],
                "gender": sequence_df.iloc[i]["gender"],
                "age": sequence_df.iloc[i]["age"],
                "age_group": sequence_df.iloc[i]["age_group"],
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "feature_rows": history["feature_row"].tolist(),
                "visual_missing_flags": visual_missing_flags.tolist(),
                "motion_missing_flags": motion_missing_flags.tolist(),
                "hr_missing_flags": hr_missing_flags.tolist(),
                "sensor_missing_flags": sensor_missing_flags.tolist(),
                "target": float(target),
            }

            # Store every scaled sensor as its own temporal column. Each value is
            # a length-SEQUENCE_LENGTH list, e.g. train_df["heart_rate"].iloc[0].
            for sensor_col in sensor_cols:
                sequence_record[sensor_col] = (
                    history[sensor_col]
                    .astype(np.float32)
                    .to_numpy(dtype=np.float32)
                    .tolist()
                )

            sequences.append(sequence_record)

    return pd.DataFrame(sequences)

In [15]:
train_df = create_temporal_sequences(train_frames)
val_df = create_temporal_sequences(val_frames)
test_df = create_temporal_sequences(test_frames)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
})

,split,sequences,subjects,target_mean
0,train,75189,39,2.971392
1,val,17208,9,2.948236
2,test,15894,9,2.881307


### Train Test Split

In [16]:
class MultimodalFusionDataset(Dataset):

    def __init__(self, sequence_df, feature_store_path):
        self.df = sequence_df.reset_index(drop=True)
        self.feature_store_path = feature_store_path
        self.feature_store = None

    def __len__(self):
        return len(self.df)

    def _features(self):
        if self.feature_store is None:
            self.feature_store = np.load(self.feature_store_path, mmap_mode="r")
        return self.feature_store

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64)
        visual_features = np.zeros((len(feature_rows), VISUAL_FEATURE_DIM), dtype=np.float32)

        valid = feature_rows >= 0
        visual_features[valid] = self._features()[feature_rows[valid]]

        # Reconstruct the T x sensor_dim tensor from separate temporal sensor
        # columns instead of reading one packed sensor_values column.
        sensors = np.stack(
            [
                np.asarray(row[sensor_col], dtype=np.float32)
                for sensor_col in sensor_cols
            ],
            axis=1
        )
        sensors = torch.tensor(sensors, dtype=torch.float32)
        sensors = torch.nan_to_num(sensors, nan=0.0, posinf=0.0, neginf=0.0)

        motion = sensors[:, motion_indices]
        motion_missing = torch.tensor(row["motion_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        motion = torch.cat([motion, motion_missing], dim=-1)

        if hr_indices:
            heart_rate = sensors[:, hr_indices]
        else:
            heart_rate = torch.zeros((sensors.shape[0], 0), dtype=torch.float32)
        hr_missing = torch.tensor(row["hr_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        heart_rate = torch.cat([heart_rate, hr_missing], dim=-1)

        visual_features = torch.tensor(visual_features, dtype=torch.float32)
        visual_missing_flags = torch.tensor(row["visual_missing_flags"], dtype=torch.float32)
        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)

        return visual_features, motion, heart_rate, visual_missing_flags, target, sample_weight, idx

In [17]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

,bin,train_count,weight
0,"(1.999, 2.5]",13031,1.140073
1,"(2.5, 3.0]",30614,0.743809
2,"(3.0, 3.5]",13128,1.135853
3,"(3.5, 4.75]",4210,2.005766


,split,sequences,subjects,target_mean,mean_sample_weight
0,train,60983,33,2.946022,1.000000
1,val,23908,12,2.999916,1.002191
2,test,23142,12,2.940087,1.023525


In [18]:
train_df.shape, val_df.shape, test_df.shape

((60983, 23), (23908, 23), (23142, 23))

In [19]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

(232, 89, 86)

In [20]:
train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
test_dataset = MultimodalFusionDataset(test_df, feature_store_path)

if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

### Training

In [21]:
class ResidualMultimodalFusionTransformer(nn.Module):

    def __init__(
        self,
        motion_input_dim,
        hr_input_dim,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.visual_projection = nn.Sequential(
            nn.Linear(visual_feature_dim + 1, visual_dim),
            nn.LayerNorm(visual_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.motion_projection = nn.Sequential(
            nn.Linear(motion_input_dim, motion_dim),
            nn.LayerNorm(motion_dim),
            nn.GELU(),
        )
        self.hr_projection = nn.Sequential(
            nn.Linear(hr_input_dim, hr_dim),
            nn.LayerNorm(hr_dim),
            nn.GELU(),
        )

        # Each modality receives capacity proportional to its input complexity.
        self.visual_position = nn.Parameter(torch.zeros(max_seq_len, visual_dim))
        self.motion_position = nn.Parameter(torch.zeros(max_seq_len, motion_dim))
        self.hr_position = nn.Parameter(torch.zeros(max_seq_len, hr_dim))

        visual_layer = nn.TransformerEncoderLayer(
            d_model=visual_dim,
            nhead=4,
            dim_feedforward=256,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        motion_layer = nn.TransformerEncoderLayer(
            d_model=motion_dim,
            nhead=4,
            dim_feedforward=64,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        hr_layer = nn.TransformerEncoderLayer(
            d_model=hr_dim,
            nhead=2,
            dim_feedforward=32,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.visual_encoder = nn.TransformerEncoder(visual_layer, num_layers=2)
        self.motion_encoder = nn.TransformerEncoder(motion_layer, num_layers=1)
        self.hr_encoder = nn.TransformerEncoder(hr_layer, num_layers=1)
        self.visual_norm = nn.LayerNorm(visual_dim)
        self.motion_norm = nn.LayerNorm(motion_dim)
        self.hr_norm = nn.LayerNorm(hr_dim)

        self.visual_regressor = nn.Sequential(
            nn.Linear(visual_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        fusion_dim = visual_dim + motion_dim + hr_dim
        self.sensor_dropout = nn.Dropout(0.25)
        self.residual_gate = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        self.sensor_residual = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.max_sensor_correction = max_sensor_correction

        # Begin as a visual-only Transformer. Sensor influence must be learned.
        nn.init.zeros_(self.sensor_residual[-1].weight)
        nn.init.zeros_(self.sensor_residual[-1].bias)
        nn.init.constant_(self.residual_gate[-2].bias, -2.0)

    @staticmethod
    def causal_mask(sequence_length, device):
        positions = torch.arange(sequence_length, device=device)
        return positions.unsqueeze(0) > positions.unsqueeze(1)

    def forward(self, visual_features, motion, heart_rate, visual_missing_flags):
        _, sequence_length, _ = visual_features.shape
        mask = self.causal_mask(sequence_length, visual_features.device)

        visual_missing_flags = visual_missing_flags.unsqueeze(-1)
        visual_input = torch.cat([visual_features, visual_missing_flags], dim=-1)
        visual_tokens = (
            self.visual_projection(visual_input)
            + self.visual_position[:sequence_length].unsqueeze(0)
        )
        motion_tokens = (
            self.motion_projection(motion)
            + self.motion_position[:sequence_length].unsqueeze(0)
        )
        hr_tokens = (
            self.hr_projection(heart_rate)
            + self.hr_position[:sequence_length].unsqueeze(0)
        )

        visual_encoded = self.visual_encoder(visual_tokens, mask=mask)
        motion_encoded = self.motion_encoder(motion_tokens, mask=mask)
        hr_encoded = self.hr_encoder(hr_tokens, mask=mask)

        visual_summary = self.visual_norm(visual_encoded[:, -1, :])
        motion_summary = self.motion_norm(motion_encoded[:, -1, :])
        hr_summary = self.hr_norm(hr_encoded[:, -1, :])
        visual_prediction = self.visual_regressor(visual_summary).squeeze(1)

        fusion_context = torch.cat(
            [visual_summary, motion_summary, hr_summary],
            dim=1,
        )
        fusion_context = self.sensor_dropout(fusion_context)
        gate = self.residual_gate(fusion_context).squeeze(1)
        residual = torch.tanh(self.sensor_residual(fusion_context).squeeze(1))
        return visual_prediction + self.max_sensor_correction * gate * residual


### Subject-level gender MAE-gap regularization

For each selected subject $s$, the mean absolute error is

$$
E_s
=
\frac{1}{N_s}\sum_{i \in s}|y_i-\hat{y}_i|.
$$

The subject errors are averaged equally within each gender:

$$
E_f=\frac{1}{|S_f|}\sum_{s\in S_f}E_s,
\qquad
E_m=\frac{1}{|S_m|}\sum_{s\in S_m}E_s.
$$

The fairness regularizer directly penalizes their absolute difference:

$$
\mathcal{L}_{gap}
=
\left|E_f-E_m\right|.
$$

The final training objective is

$$
\mathcal{L}
=
\mathcal{L}_{MSE}
+\lambda\mathcal{L}_{gap}.
$$

Each training batch samples equal numbers of subjects from both genders and
multiple windows per subject. Consequently, each subject contributes equally
to the regularizer regardless of their number of available temporal windows.
The value of $\lambda$ and the retained checkpoint are selected using validation
gender gap while constraining the permitted increase in overall validation MAE.


In [22]:
import copy
import random
from importlib import reload
import src.evaluation as ev

ev = reload(ev)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_model():
    return ResidualMultimodalFusionTransformer(
        motion_input_dim=MOTION_INPUT_DIM,
        hr_input_dim=HR_INPUT_DIM,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    )


class SubjectGenderBatchSampler(torch.utils.data.Sampler):
    """Sample equal subjects per gender and multiple windows per subject."""

    def __init__(self, sequence_df, subjects_per_gender, windows_per_subject, seed):
        self.subjects_per_gender = int(subjects_per_gender)
        self.windows_per_subject = int(windows_per_subject)
        self.seed = int(seed)
        self.epoch = 0
        self.gender_subject_indices = {}
        frame = sequence_df.reset_index(drop=True)
        for (gender, subject), rows in frame.groupby(
            ["gender", "subject_id"], observed=True, sort=True
        ):
            if pd.isna(gender) or pd.isna(subject) or str(gender) == "__missing__":
                continue
            self.gender_subject_indices.setdefault(str(gender), {})[str(subject)] = (
                rows.index.to_numpy(dtype=int)
            )
        self.genders = sorted(self.gender_subject_indices)
        if len(self.genders) != 2:
            raise ValueError("Subject-level gender gap training requires exactly two gender groups.")
        self.batch_size = len(self.genders) * self.subjects_per_gender * self.windows_per_subject
        self.num_batches = int(np.ceil(len(frame) / self.batch_size))

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        for _ in range(self.num_batches):
            batch = []
            for gender in self.genders:
                subject_map = self.gender_subject_indices[gender]
                subjects = list(subject_map)
                selected = rng.choice(
                    subjects,
                    size=self.subjects_per_gender,
                    replace=len(subjects) < self.subjects_per_gender,
                )
                for subject in selected:
                    indices = subject_map[subject]
                    batch.extend(
                        rng.choice(
                            indices,
                            size=self.windows_per_subject,
                            replace=len(indices) < self.windows_per_subject,
                        ).astype(int).tolist()
                    )
            rng.shuffle(batch)
            yield batch


def make_train_loader(run_seed):
    return DataLoader(
        train_dataset,
        batch_sampler=SubjectGenderBatchSampler(
            train_dataset.df,
            subjects_per_gender=SUBJECTS_PER_GENDER_PER_BATCH,
            windows_per_subject=WINDOWS_PER_SUBJECT,
            seed=run_seed,
        ),
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=4,
    )


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def build_group_codes(sequence_df, attribute):
    values = sequence_df[attribute].astype(str)
    groups = sorted(value for value in values.unique() if value != "__missing__")
    mapping = {group: code for code, group in enumerate(groups)}
    codes = values.map(mapping).fillna(-1).astype(int).to_numpy()
    return torch.tensor(codes, dtype=torch.long), mapping


GENDER_CODES, GENDER_MAP = build_group_codes(train_dataset.df, "gender")
SUBJECT_CODES, SUBJECT_MAP = build_group_codes(train_dataset.df, "subject_id")
print("Gender groups used by the regularizer:", GENDER_MAP)


def subject_gender_mae_gap_loss(per_sample_mae, gender_codes, subject_codes):
    gender_subject_maes = {}
    for subject_code in torch.unique(subject_codes):
        code = int(subject_code.item())
        if code < 0:
            continue
        subject_mask = subject_codes == subject_code
        subject_gender = int(gender_codes[subject_mask][0].item())
        if subject_gender < 0:
            continue
        gender_subject_maes.setdefault(subject_gender, []).append(
            per_sample_mae[subject_mask].mean()
        )
    if len(gender_subject_maes) != 2:
        return per_sample_mae.new_zeros(()), False
    gender_maes = [
        torch.stack(subject_maes).mean()
        for _gender, subject_maes in sorted(gender_subject_maes.items())
    ]
    return torch.abs(gender_maes[0] - gender_maes[1]), True


def train_one_epoch_subject_gender_gap(model, loader, optimizer, criterion, fairness_lambda, epoch):
    model.train()
    totals = {"loss": 0.0, "task_loss": 0.0, "gap_loss": 0.0}
    preds_all, labels_all = [], []
    active_batches = 0

    for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, idx in tqdm(loader, desc="Training", leave=False):
        visual_features = visual_features.to(device)
        motion = motion.to(device)
        heart_rate = heart_rate.to(device)
        visual_missing_flags = visual_missing_flags.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(visual_features, motion, heart_rate, visual_missing_flags)
        per_sample_mse = criterion(preds, labels)
        task_loss = weighted_task_loss(per_sample_mse, sample_weights)
        cpu_idx = idx.detach().cpu().long()
        gender_codes = GENDER_CODES[cpu_idx].to(device)
        subject_codes = SUBJECT_CODES[cpu_idx].to(device)
        gap_loss, gap_active = subject_gender_mae_gap_loss(
            torch.abs(preds - labels),
            gender_codes,
            subject_codes,
        )
        active_batches += int(gap_active)
        loss = task_loss + fairness_lambda * gap_loss if epoch >= FAIRNESS_WARMUP_EPOCHS else task_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        totals["loss"] += float(loss.item())
        totals["task_loss"] += float(task_loss.item())
        totals["gap_loss"] += float(gap_loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        **{key: value / len(loader) for key, value in totals.items()},
        "mae": float(mean_absolute_error(labels_all, preds_all)),
        "rmse": float(np.sqrt(mean_squared_error(labels_all, preds_all))),
        "gap_active_batch_rate": active_batches / max(len(loader), 1),
    }


def compute_subject_gender_metrics(frame):
    subject_errors = (
        frame.assign(abs_error=np.abs(frame["pred"].astype(float) - frame["true"].astype(float)))
        .groupby(["subject_id", "gender"], observed=True)["abs_error"]
        .mean()
        .reset_index()
    )
    gender_mae = subject_errors.groupby("gender", observed=True)["abs_error"].mean()
    return gender_mae, float(gender_mae.max()), float(gender_mae.max() - gender_mae.min())


def evaluate_loader(model, loader, sequence_df):
    model.eval()
    preds_all, labels_all, indices_all = [], [], []
    with torch.no_grad():
        for visual_features, motion, heart_rate, visual_missing_flags, labels, _sample_weights, idx in tqdm(loader, desc="Evaluating", leave=False):
            preds = model(
                visual_features.to(device),
                motion.to(device),
                heart_rate.to(device),
                visual_missing_flags.to(device),
            )
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())
            indices_all.extend(idx.detach().cpu().numpy())

    frame = sequence_df.iloc[np.asarray(indices_all, dtype=int)][
        ["subject_experiment_id", "subject_id", "time_sec", "attention_bin", "gender", "age", "age_group"]
    ].reset_index(drop=True)
    frame.insert(0, "true", np.asarray(labels_all, dtype=float))
    frame.insert(0, "pred", np.asarray(preds_all, dtype=float))
    frame = ev.add_robustness_metadata(frame, sequence_df, visual_missing_col="visual_missing_flags")

    overall = ev.compute_prediction_metrics(frame)
    age_mae, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    gender_mae, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    subject_gender_mae, subject_gender_worst, subject_gender_gap = compute_subject_gender_metrics(frame)
    return {
        "mae": overall["mae"], "rmse": overall["rmse"], "r2": overall["r2"],
        "true_mean": overall["true_mean"], "pred_mean": overall["pred_mean"],
        "age_worst_group_mae": age_worst, "age_gap": age_gap,
        "gender_worst_group_mae": gender_worst, "gender_gap": gender_gap,
        "subject_gender_worst_group_mae": subject_gender_worst,
        "subject_gender_gap": subject_gender_gap,
        "age_mae_per_group": age_mae.to_dict(),
        "gender_mae_per_group": gender_mae.to_dict(),
        "subject_gender_mae_per_group": subject_gender_mae.to_dict(),
    }, frame


Gender groups used by the regularizer: {'female': 0, 'male': 1}


In [ ]:
class EarlyStopping:
    def __init__(self, patience, model_path):
        self.patience = patience
        self.model_path = model_path
        self.best_score = float("inf")
        self.best_epoch = None
        self.counter = 0

    def step(self, score, model, epoch):
        if score < self.best_score:
            self.best_score = float(score)
            self.best_epoch = epoch
            self.counter = 0
            torch.save(copy.deepcopy(model.state_dict()), self.model_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


criterion = nn.MSELoss(reduction="none")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 7

FAIRNESS_AXIS = "gender"
FAIRNESS_LAMBDAS = [0.70]
FAIRNESS_WARMUP_EPOCHS = 2
SUBJECTS_PER_GENDER_PER_BATCH = 2
WINDOWS_PER_SUBJECT = 8
MAX_VAL_MAE_INCREASE = 0.02
FAIRNESS_MONITOR_MAE_PENALTY = 5.0

RUN_SEEDS = [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]
SELECTION_SEEDS = RUN_SEEDS

BASELINE_RESULTS_DIR = "results/Multimodal Fusion Residual Transformer"
RESULTS_DIR = "results/Multimodal Fusion Subject Gender MAE Gap Residual Transformer"
MODEL_DIR = "models/Multimodal Fusion Subject Gender MAE Gap Residual Transformer"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def lambda_tag(value):
    return str(value).replace(".", "p")


def candidate_run_name(run_seed, fairness_lambda):
    return f"subject_gender_mae_gap_residual_transformer_lambda{lambda_tag(fairness_lambda)}_seed{run_seed}"


baseline_seed_results = pd.read_csv(os.path.join(BASELINE_RESULTS_DIR, "seed_results.csv"))
baseline_subject_seed_metrics = pd.read_csv(
    os.path.join(BASELINE_RESULTS_DIR, "subject_seed_metrics.csv")
)
baseline_ensemble_path = os.path.join(
    BASELINE_RESULTS_DIR, "seed_ensemble_test_predictions.csv"
)

missing_seeds = sorted(set(RUN_SEEDS) - set(baseline_seed_results["run_seed"].astype(int)))
if missing_seeds:
    raise ValueError(f"Baseline seed_results.csv is missing seeds: {missing_seeds}")
if not os.path.exists(baseline_ensemble_path):
    raise FileNotFoundError(f"Required baseline ensemble file is missing: {baseline_ensemble_path}")

baseline_seed_results = baseline_seed_results.set_index("run_seed", drop=False)
baseline_validation_metrics = {
    int(run_seed): {
        "mae": float(row["val_mae"]),
        "gender_worst_group_mae": float(row["val_gender_worst_group_mae"]),
        "gender_gap": float(row["val_gender_gap"]),
    }
    for run_seed, row in baseline_seed_results.iterrows()
}

# Recover baseline test subject-level gender metrics from the saved per-subject MAEs.
subject_gender_lookup = test_df[["subject_id", "gender"]].drop_duplicates("subject_id")
baseline_subject_gender_table = (
    baseline_subject_seed_metrics
    .merge(subject_gender_lookup, on="subject_id", how="left", validate="many_to_one")
    .merge(
        baseline_seed_results[["run_name", "run_seed"]].reset_index(drop=True),
        on="run_name",
        how="left",
        validate="many_to_one",
    )
)
if baseline_subject_gender_table["gender"].isna().any():
    raise ValueError("Could not map every baseline test subject to a gender group.")

baseline_test_subject_gender_metrics = {}
for run_seed, group in baseline_subject_gender_table.groupby("run_seed", observed=True):
    per_gender = group.groupby("gender", observed=True)["mae"].mean()
    baseline_test_subject_gender_metrics[int(run_seed)] = {
        "mae_per_group": per_gender.to_dict(),
        "worst_group_mae": float(per_gender.max()),
        "gap": float(per_gender.max() - per_gender.min()),
    }

print(f"Using baseline aggregate results from: {BASELINE_RESULTS_DIR}")
print(f"Fixed split random state: {SPLIT_RANDOM_STATE}")
print(f"Fairness axis: {FAIRNESS_AXIS}")
print(f"Candidate lambdas: {FAIRNESS_LAMBDAS}")


In [ ]:
def train_candidate(run_seed, fairness_lambda):
    set_global_seed(run_seed)
    run_name = candidate_run_name(run_seed, fairness_lambda)
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")
    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.3, patience=3)
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader(run_seed)
    baseline_val_mae = baseline_validation_metrics[run_seed]["mae"]
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_subject_gender_gap(
            model, run_train_loader, optimizer, criterion, fairness_lambda, epoch
        )
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        mae_excess = max(0.0, val_metrics["mae"] - baseline_val_mae - MAX_VAL_MAE_INCREASE)
        monitor = val_metrics["subject_gender_gap"] + FAIRNESS_MONITOR_MAE_PENALTY * mae_excess
        scheduler.step(monitor)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_task_loss": train_metrics["task_loss"],
            "train_gap_loss": train_metrics["gap_loss"],
            "train_gap_active_batch_rate": train_metrics["gap_active_batch_rate"],
            "train_mae": train_metrics["mae"],
            "val_mae": val_metrics["mae"],
            "val_gender_worst_group_mae": val_metrics["gender_worst_group_mae"],
            "val_gender_gap": val_metrics["gender_gap"],
            "val_subject_gender_worst_group_mae": val_metrics["subject_gender_worst_group_mae"],
            "val_subject_gender_gap": val_metrics["subject_gender_gap"],
            "selection_monitor": monitor,
        })
        print(
            f"Epoch {epoch + 1:02d} | val MAE {val_metrics['mae']:.4f} | "
            f"gender worst {val_metrics['gender_worst_group_mae']:.4f} | "
            f"subject gender gap {val_metrics['subject_gender_gap']:.4f}"
        )
        if epoch >= FAIRNESS_WARMUP_EPOCHS and early_stopping.step(monitor, model, epoch):
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    val_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_path)
    return {
        "run_name": run_name, "run_seed": run_seed, "fairness_lambda": fairness_lambda,
        "best_epoch": early_stopping.best_epoch + 1, "num_epochs_run": len(history),
        "model_path": model_path, "val_prediction_path": val_path,
        "val_metrics": val_metrics, "history": history,
    }


selection_runs = [
    train_candidate(run_seed, fairness_lambda)
    for fairness_lambda in FAIRNESS_LAMBDAS
    for run_seed in SELECTION_SEEDS
]
validation_rows = []
for run in selection_runs:
    baseline = baseline_validation_metrics[run["run_seed"]]
    candidate = run["val_metrics"]
    validation_rows.append({
        "run_seed": run["run_seed"], "fairness_lambda": run["fairness_lambda"],
        "baseline_val_mae": baseline["mae"], "candidate_val_mae": candidate["mae"],
        "val_mae_increase": candidate["mae"] - baseline["mae"],
        "baseline_val_gender_worst_group_mae": baseline["gender_worst_group_mae"],
        "candidate_val_gender_worst_group_mae": candidate["gender_worst_group_mae"],
        "gender_worst_group_gain": baseline["gender_worst_group_mae"] - candidate["gender_worst_group_mae"],
        "baseline_val_gender_gap": baseline["gender_gap"],
        "candidate_val_gender_gap": candidate["gender_gap"],
        "gender_gap_reduction": baseline["gender_gap"] - candidate["gender_gap"],
        "candidate_val_subject_gender_worst_group_mae": candidate["subject_gender_worst_group_mae"],
        "candidate_val_subject_gender_gap": candidate["subject_gender_gap"],
        "within_mae_constraint": candidate["mae"] <= baseline["mae"] + MAX_VAL_MAE_INCREASE,
    })

validation_grid = pd.DataFrame(validation_rows)
validation_summary = (
    validation_grid.groupby("fairness_lambda", observed=True)
    .agg(
        seeds=("run_seed", "nunique"),
        mean_val_mae_increase=("val_mae_increase", "mean"),
        max_val_mae_increase=("val_mae_increase", "max"),
        mae_constraint_rate=("within_mae_constraint", "mean"),
        mean_gender_gap_reduction=("gender_gap_reduction", "mean"),
        gender_gap_reduction_positive_rate=("gender_gap_reduction", lambda x: float((x > 0).mean())),
        mean_candidate_subject_gender_gap=("candidate_val_subject_gender_gap", "mean"),
        mean_candidate_subject_gender_worst_group_mae=("candidate_val_subject_gender_worst_group_mae", "mean"),
    ).reset_index()
)
eligible = validation_summary[
    validation_summary["mean_val_mae_increase"] <= MAX_VAL_MAE_INCREASE
].copy()
if eligible.empty:
    print("WARNING: no lambda satisfied the mean validation MAE constraint.")
    eligible = validation_summary.copy()

# Baseline validation subject-level predictions were not retained. Since the
# same baseline applies to every lambda, minimizing candidate subject-gender gap
# gives the same lambda ordering as maximizing its reduction from that baseline.
selected_row = eligible.sort_values(
    ["mean_candidate_subject_gender_gap", "mean_gender_gap_reduction",
     "mean_candidate_subject_gender_worst_group_mae", "mean_val_mae_increase"],
    ascending=[True, False, True, True],
).iloc[0]
SELECTED_LAMBDA = float(selected_row["fairness_lambda"])
display(validation_grid.round(4))
display(validation_summary.round(4))
print(f"Selected subject-level gender MAE-gap lambda: {SELECTED_LAMBDA}")

selected_runs = [run for run in selection_runs if run["fairness_lambda"] == SELECTED_LAMBDA]
for run_seed in RUN_SEEDS:
    if run_seed not in SELECTION_SEEDS:
        selected_runs.append(train_candidate(run_seed, SELECTED_LAMBDA))
print("Finished all selected subject-level gender MAE-gap runs.")


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== subject_gender_mae_gap_residual_transformer_lambda0p7_seed42 ===


Epoch 01 | val MAE 0.3445 | gender worst 0.3582 | subject gender gap 0.0238


Epoch 02 | val MAE 0.3448 | gender worst 0.3645 | subject gender gap 0.0057


Epoch 03 | val MAE 0.3361 | gender worst 0.3508 | subject gender gap 0.0184


...

Epoch 05 | val MAE 0.3211 | gender worst 0.3244 | subject gender gap 0.0289


Epoch 06 | val MAE 0.3390 | gender worst 0.3398 | subject gender gap 0.0522


Epoch 07 | val MAE 0.3398 | gender worst 0.3408 | subject gender gap 0.0468


Epoch 08 | val MAE 0.3341 | gender worst 0.3371 | subject gender gap 0.0536


Training:  82%|████████▏ | 1570/1906 [00:26<00:05, 59.57it/s]

In [ ]:
test_predictions = {}
test_rows = []
for run in selected_runs:
    model = make_model().to(device)
    model.load_state_dict(torch.load(run["model_path"], map_location=device, weights_only=True))
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run["run_name"]] = test_frame
    test_path = os.path.join(RESULTS_DIR, f"{run['run_name']}_test_predictions.csv")
    ev.save_prediction_frame(test_frame, test_path)
    run["test_prediction_path"] = test_path
    run["test_metrics"] = test_metrics
    row = {
        "run_name": run["run_name"], "run_seed": run["run_seed"],
        "fairness_lambda": run["fairness_lambda"], "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
    }
    row.update({f"val_{k}": v for k, v in run["val_metrics"].items() if not isinstance(v, dict)})
    row.update({f"test_{k}": v for k, v in test_metrics.items() if not isinstance(v, dict)})
    test_rows.append(row)

seed_results = pd.DataFrame(test_rows).sort_values("run_seed")
summary_metrics = [
    "test_mae", "test_rmse", "test_r2", "test_age_worst_group_mae",
    "test_age_gap", "test_gender_worst_group_mae", "test_gender_gap",
    "test_subject_gender_worst_group_mae", "test_subject_gender_gap",
]
seed_summary = seed_results[summary_metrics].agg(["mean", "std", "min", "median", "max"]).T.reset_index(names="metric")
display(seed_results.round(4))
display(seed_summary.round(4))


In [ ]:
# Compare against the saved baseline aggregate metrics. Per-window baseline
# predictions were not retained, so these are seed-matched aggregate differences,
# not paired-window comparisons.
comparison_rows = []
subject_gender_rows = []
for run in selected_runs:
    run_seed = run["run_seed"]
    baseline = baseline_seed_results.loc[run_seed]
    candidate = run["test_metrics"]
    baseline_subject = baseline_test_subject_gender_metrics[run_seed]
    _, candidate_subject_worst, candidate_subject_gap = compute_subject_gender_metrics(
        test_predictions[run["run_name"]]
    )

    subject_gender_rows.append({
        "run_seed": run_seed,
        "baseline_subject_gender_worst_group_mae": baseline_subject["worst_group_mae"],
        "candidate_subject_gender_worst_group_mae": candidate_subject_worst,
        "subject_gender_worst_group_gain": baseline_subject["worst_group_mae"] - candidate_subject_worst,
        "baseline_subject_gender_gap": baseline_subject["gap"],
        "candidate_subject_gender_gap": candidate_subject_gap,
        "subject_gender_gap_reduction": baseline_subject["gap"] - candidate_subject_gap,
        "baseline_subject_gender_mae_per_group": baseline_subject["mae_per_group"],
    })

    for attribute in ["gender", "age_group"]:
        prefix = "test_gender" if attribute == "gender" else "test_age"
        candidate_prefix = "gender" if attribute == "gender" else "age"
    
        comparison_rows.append({
            "run_seed": run_seed,
            "attribute": attribute,
            "baseline_mae": float(baseline["test_mae"]),
            "candidate_mae": float(candidate["mae"]),
            "overall_mae_gain": float(baseline["test_mae"] - candidate["mae"]),
            "baseline_worst_group_mae": float(baseline[f"{prefix}_worst_group_mae"]),
            "candidate_worst_group_mae": float(candidate[f"{candidate_prefix}_worst_group_mae"]),
            "worst_group_mae_gain": float(
                baseline[f"{prefix}_worst_group_mae"]
                - candidate[f"{candidate_prefix}_worst_group_mae"]
            ),
            "baseline_gap": float(baseline[f"{prefix}_gap"]),
            "candidate_gap": float(candidate[f"{candidate_prefix}_gap"]),
            "gap_reduction": float(
                baseline[f"{prefix}_gap"] - candidate[f"{candidate_prefix}_gap"]
            ),
        })

fairness_comparison = pd.DataFrame(comparison_rows)
fairness_seed_summary = fairness_comparison.groupby("attribute", observed=True).agg(
    seeds=("run_seed", "nunique"),
    mean_overall_mae_gain=("overall_mae_gain", "mean"),
    mean_worst_group_mae_gain=("worst_group_mae_gain", "mean"),
    worst_group_gain_positive_rate=("worst_group_mae_gain", lambda x: float((x > 0).mean())),
    mean_gap_reduction=("gap_reduction", "mean"),
    gap_reduction_positive_rate=("gap_reduction", lambda x: float((x > 0).mean())),
).reset_index()

subject_gender_comparison = pd.DataFrame(subject_gender_rows)
subject_gender_summary = pd.DataFrame([{
    "seeds": subject_gender_comparison["run_seed"].nunique(),
    "mean_subject_gender_worst_group_gain": subject_gender_comparison["subject_gender_worst_group_gain"].mean(),
    "subject_gender_worst_group_gain_positive_rate": float((subject_gender_comparison["subject_gender_worst_group_gain"] > 0).mean()),
    "mean_subject_gender_gap_reduction": subject_gender_comparison["subject_gender_gap_reduction"].mean(),
    "subject_gender_gap_reduction_positive_rate": float((subject_gender_comparison["subject_gender_gap_reduction"] > 0).mean()),
}])
display(fairness_seed_summary.round(4))
display(subject_gender_summary.round(4))


def attention_bin_subgroup_mae(frame, attribute):
    """Window-level and equally weighted subject-level MAE within each bin/group."""
    working = frame.copy()
    working["abs_error"] = np.abs(
        working["pred"].astype(float) - working["true"].astype(float)
    )
    window_table = (
        working.groupby(["attention_bin", attribute], observed=True)
        .agg(n_samples=("abs_error", "size"), n_subjects=("subject_id", "nunique"),
             window_mae=("abs_error", "mean"))
        .reset_index()
    )
    subject_errors = (
        working.groupby(["attention_bin", attribute, "subject_id"], observed=True)["abs_error"]
        .mean().reset_index()
    )
    subject_table = (
        subject_errors.groupby(["attention_bin", attribute], observed=True)
        .agg(subject_balanced_mae=("abs_error", "mean")).reset_index()
    )
    return window_table.merge(subject_table, on=["attention_bin", attribute], how="left")


def attention_bin_gap_table(subgroup_table, attribute):
    rows = []
    for attention_bin, group in subgroup_table.groupby("attention_bin", observed=True):
        if group[attribute].nunique() < 2:
            continue
        rows.append({
            "attention_bin": attention_bin,
            "groups_present": int(group[attribute].nunique()),
            "window_mae_gap": float(group["window_mae"].max() - group["window_mae"].min()),
            "subject_balanced_mae_gap": float(
                group["subject_balanced_mae"].max() - group["subject_balanced_mae"].min()
            ),
        })
    return pd.DataFrame(rows)


attention_bin_subgroup_rows = []
attention_bin_gap_rows = []
for run in selected_runs:
    run_seed = run["run_seed"]
    frame = test_predictions[run["run_name"]]
    for attribute in ["gender", "age_group"]:
        subgroup_table = attention_bin_subgroup_mae(frame, attribute)
        subgroup_table.insert(0, "attribute", attribute)
        subgroup_table.insert(0, "model", "subject_gender_gap")
        subgroup_table.insert(0, "run_seed", run_seed)
        attention_bin_subgroup_rows.append(subgroup_table)
        gap_table = attention_bin_gap_table(subgroup_table, attribute)
        gap_table.insert(0, "attribute", attribute)
        gap_table.insert(0, "model", "subject_gender_gap")
        gap_table.insert(0, "run_seed", run_seed)
        attention_bin_gap_rows.append(gap_table)

attention_bin_subgroup_mae_table = pd.concat(attention_bin_subgroup_rows, ignore_index=True)
attention_bin_gap_comparison = pd.concat(attention_bin_gap_rows, ignore_index=True)
display(
    attention_bin_subgroup_mae_table[
        attention_bin_subgroup_mae_table["attribute"] == "gender"
    ].round(4)
)


In [ ]:
def aligned_ensemble(frames):
    keys = ["subject_experiment_id", "time_sec"]
    first = frames[0].copy().sort_values(keys).reset_index(drop=True)
    predictions = []
    for frame in frames:
        aligned = frame.sort_values(keys).reset_index(drop=True)
        if not first[keys].equals(aligned[keys]):
            raise ValueError("Prediction frames are not aligned.")
        predictions.append(aligned["pred"].to_numpy(dtype=float))
    first["pred"] = np.mean(predictions, axis=0)
    return first


candidate_ensemble = aligned_ensemble(list(test_predictions.values()))
baseline_ensemble = pd.read_csv(baseline_ensemble_path)
ensemble_rows = []
for model_name, frame in [
    ("unregularized residual Transformer", baseline_ensemble),
    ("subject-level gender MAE-gap regularized residual Transformer", candidate_ensemble),
]:
    metrics = ev.compute_prediction_metrics(frame)
    _, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    _, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    _, subject_gender_worst, subject_gender_gap = compute_subject_gender_metrics(frame)
    ensemble_rows.append({
        "model": model_name, **metrics, "age_worst_group_mae": age_worst,
        "age_gap": age_gap, "gender_worst_group_mae": gender_worst, "gender_gap": gender_gap,
        "subject_gender_worst_group_mae": subject_gender_worst,
        "subject_gender_gap": subject_gender_gap,
    })
ensemble_comparison = pd.DataFrame(ensemble_rows)

ensemble_attention_bin_rows = []
ensemble_attention_bin_gap_rows = []
for model_name, frame in [
    ("baseline", baseline_ensemble),
    ("subject_gender_gap", candidate_ensemble),
]:
    for attribute in ["gender", "age_group"]:
        subgroup_table = attention_bin_subgroup_mae(frame, attribute)
        subgroup_table.insert(0, "attribute", attribute)
        subgroup_table.insert(0, "model", model_name)
        ensemble_attention_bin_rows.append(subgroup_table)
        gap_table = attention_bin_gap_table(subgroup_table, attribute)
        gap_table.insert(0, "attribute", attribute)
        gap_table.insert(0, "model", model_name)
        ensemble_attention_bin_gap_rows.append(gap_table)

ensemble_attention_bin_subgroup_mae = pd.concat(ensemble_attention_bin_rows, ignore_index=True)
ensemble_attention_bin_subgroup_gaps = pd.concat(ensemble_attention_bin_gap_rows, ignore_index=True)

BOOTSTRAP_RUNS = 1000
bootstrap_summary, _bootstrap_samples = ev.bootstrap_table(
    test_predictions, cluster_col="subject_id", n_boot=BOOTSTRAP_RUNS, seed=SEED,
)
display(ensemble_comparison.round(4))
display(
    ensemble_attention_bin_subgroup_mae[
        ensemble_attention_bin_subgroup_mae["attribute"] == "gender"
    ].round(4)
)
display(bootstrap_summary.round(4))


In [ ]:
validation_grid.to_csv(os.path.join(RESULTS_DIR, "validation_lambda_grid.csv"), index=False)
validation_summary.to_csv(os.path.join(RESULTS_DIR, "validation_lambda_summary.csv"), index=False)
seed_results.to_csv(os.path.join(RESULTS_DIR, "selected_seed_results.csv"), index=False)
seed_summary.to_csv(os.path.join(RESULTS_DIR, "selected_seed_summary.csv"), index=False)
fairness_comparison.to_csv(os.path.join(RESULTS_DIR, "aggregate_fairness_comparison.csv"), index=False)
fairness_seed_summary.to_csv(os.path.join(RESULTS_DIR, "fairness_seed_summary.csv"), index=False)
subject_gender_comparison.to_csv(os.path.join(RESULTS_DIR, "subject_gender_comparison.csv"), index=False)
subject_gender_summary.to_csv(os.path.join(RESULTS_DIR, "subject_gender_summary.csv"), index=False)
attention_bin_subgroup_mae_table.to_csv(os.path.join(RESULTS_DIR, "candidate_attention_bin_subgroup_mae.csv"), index=False)
attention_bin_gap_comparison.to_csv(os.path.join(RESULTS_DIR, "candidate_attention_bin_subgroup_gaps.csv"), index=False)
ensemble_comparison.to_csv(os.path.join(RESULTS_DIR, "ensemble_comparison.csv"), index=False)
ensemble_attention_bin_subgroup_mae.to_csv(os.path.join(RESULTS_DIR, "ensemble_attention_bin_subgroup_mae.csv"), index=False)
ensemble_attention_bin_subgroup_gaps.to_csv(os.path.join(RESULTS_DIR, "ensemble_attention_bin_subgroup_gaps.csv"), index=False)
bootstrap_summary.to_csv(os.path.join(RESULTS_DIR, "selected_subject_bootstrap.csv"), index=False)
ev.save_prediction_frame(candidate_ensemble, os.path.join(RESULTS_DIR, "subject_gender_mae_gap_seed_ensemble_test_predictions.csv"))


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


with open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w") as file:
    json.dump(json_safe({
        "experiment": "Subject-level gender MAE-gap regularized residual multimodal Transformer",
        "architecture_changed": False,
        "fairness_axis": FAIRNESS_AXIS,
        "regularizer": "range between mean subject MAEs across gender groups",
        "objective": "task MSE + lambda * subject-level gender MAE range",
        "subjects_per_gender_per_batch": SUBJECTS_PER_GENDER_PER_BATCH,
        "windows_per_subject": WINDOWS_PER_SUBJECT,
        "selected_lambda": SELECTED_LAMBDA,
        "candidate_lambdas": FAIRNESS_LAMBDAS,
        "selection_seeds": SELECTION_SEEDS,
        "run_seeds": RUN_SEEDS,
        "max_validation_mae_increase": MAX_VAL_MAE_INCREASE,
        "split_random_state": SPLIT_RANDOM_STATE,
        "baseline_comparison_source": "aggregate per-seed summaries and saved test ensemble",
        "paired_per_window_baseline_comparison_available": False,
        "runs": selected_runs,
    }), file, indent=2, default=str)
print(f"Saved subject-level gender MAE-gap results to: {RESULTS_DIR}")


### Reading the results

The primary success metric is the reduction in **gender MAE gap** relative to
the unregularized residual Transformer. Overall MAE and gender worst-group MAE
must also be reported to determine whether a smaller gap reflects a reasonable
fairness-performance trade-off or merely worsening of the better-performing
group. The training regularizer weights each selected subject equally. Age-group
results are secondary auditing metrics.

`attention_bin_subgroup_mae.csv` reports subgroup MAE separately within every
attention bin for each seed. `window_mae` weights every temporal observation
equally, while `subject_balanced_mae` first calculates each subject's MAE within
the bin and then weights subjects equally. The corresponding gap-comparison
table shows whether the fairness-aware model reduces subgroup disparities
within particular attention-score ranges.


### Repeated subject-split generalizability evaluation

The fixed-split multi-seed experiment measures sensitivity to model
initialization for one selection of participants. This additional evaluation
instead measures sensitivity to participant selection. Ten distinct,
demographically constrained 60--20--20 subject-level train-validation-test splits are
paired with ten training seeds. The exact same split-seed pairs are used by the
unregularized, gender-regularized, and age-regularized models.

Fairness regularization strengths are fixed to the values selected in the
original fixed-split validation experiments. They are not re-selected for each
new split. Consequently, the repeated-split test results evaluate whether the
previously selected interventions generalize to different held-out subjects.


In [ ]:
# Load the exact split-seed pairs generated by the unregularized notebook.
REPEATED_SPLIT_BASELINE_DIR = "results/Multimodal Fusion Residual Transformer Repeated Subject Splits"
REPEATED_SPLIT_MANIFEST_PATH = os.path.join(REPEATED_SPLIT_BASELINE_DIR, "split_manifest.csv")
REPEATED_SPLIT_BASELINE_RESULTS_PATH = os.path.join(
    REPEATED_SPLIT_BASELINE_DIR, "repeated_split_results.csv"
)
REPEATED_SPLIT_RESULTS_DIR = "results/Multimodal Fusion Subject Gender MAE Gap Repeated Subject Splits"
REPEATED_SPLIT_MODEL_DIR = "models/Multimodal Fusion Subject Gender MAE Gap Repeated Subject Splits"
REPEATED_SPLIT_FAIRNESS_LAMBDA = 0.70
os.makedirs(REPEATED_SPLIT_RESULTS_DIR, exist_ok=True)
os.makedirs(REPEATED_SPLIT_MODEL_DIR, exist_ok=True)

if not os.path.exists(REPEATED_SPLIT_MANIFEST_PATH):
    raise FileNotFoundError(
        "Run the repeated-split section of the unregularized residual Transformer "
        f"notebook first. Missing: {REPEATED_SPLIT_MANIFEST_PATH}"
    )
if not os.path.exists(REPEATED_SPLIT_BASELINE_RESULTS_PATH):
    raise FileNotFoundError(
        "Run the repeated-split unregularized models first. Missing: "
        f"{REPEATED_SPLIT_BASELINE_RESULTS_PATH}"
    )

repeated_split_manifest = pd.read_csv(REPEATED_SPLIT_MANIFEST_PATH)
repeated_split_baseline_results = pd.read_csv(REPEATED_SPLIT_BASELINE_RESULTS_PATH)
if repeated_split_manifest["split_id"].nunique() != 10:
    raise ValueError("The shared repeated-split manifest must contain ten splits.")


def activate_repeated_split(split_id):
    """Activate one shared split and refresh fairness-regularizer group codes."""
    global train_df, val_df, test_df
    global train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader
    global GENDER_CODES, SUBJECT_CODES, GENDER_MAP, SUBJECT_MAP

    current = repeated_split_manifest[repeated_split_manifest["split_id"] == split_id]
    subject_sets = {
        split_name: set(current.loc[current["split"] == split_name, "subject_id"])
        for split_name in ["train", "validation", "test"]
    }
    if (
        subject_sets["train"] & subject_sets["validation"]
        or subject_sets["train"] & subject_sets["test"]
        or subject_sets["validation"] & subject_sets["test"]
    ):
        raise ValueError(f"Subject overlap detected in repeated split {split_id}.")

    frame_splits = {
        split_name: temporal_frame_dataset[
            temporal_frame_dataset["subject_id"].isin(subject_ids)
        ].copy()
        for split_name, subject_ids in subject_sets.items()
    }
    repeated_sensor_means = frame_splits["train"].loc[
        frame_splits["train"]["sensor_missing"] == 0, sensor_cols
    ].mean().fillna(0.0)
    repeated_sensor_stds = frame_splits["train"].loc[
        frame_splits["train"]["sensor_missing"] == 0, sensor_cols
    ].std().replace(0, np.nan).fillna(1.0)

    def scale(frame):
        frame = frame.copy()
        scaled = ((frame[sensor_cols] - repeated_sensor_means) / repeated_sensor_stds)
        scaled = scaled.astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        scaled.loc[frame["sensor_missing"] == 1, :] = 0.0
        frame.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
        return frame

    train_df = create_temporal_sequences(scale(frame_splits["train"]))
    val_df = create_temporal_sequences(scale(frame_splits["validation"]))
    test_df = create_temporal_sequences(scale(frame_splits["test"]))
    train_df, val_df, test_df, _ = add_attention_bin_weights(
        WEIGHT_ALPHA, train_df, val_df, test_df
    )
    train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
    val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
    test_dataset = MultimodalFusionDataset(test_df, feature_store_path)
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=4,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=4,
    )
    GENDER_CODES, GENDER_MAP = build_group_codes(train_dataset.df, "gender")
    SUBJECT_CODES, SUBJECT_MAP = build_group_codes(train_dataset.df, "subject_id")


def train_repeated_split_candidate(split_id, run_seed, baseline_val_mae):
    set_global_seed(run_seed)
    activate_repeated_split(split_id)
    run_name = (
        f"subject_gender_mae_gap_repeated_split{split_id:02d}_"
        f"lambda{lambda_tag(REPEATED_SPLIT_FAIRNESS_LAMBDA)}_seed{run_seed}"
    )
    model_path = os.path.join(REPEATED_SPLIT_MODEL_DIR, f"{run_name}.pt")
    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader(run_seed)
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_subject_gender_gap(
            model, run_train_loader, optimizer, criterion,
            REPEATED_SPLIT_FAIRNESS_LAMBDA, epoch,
        )
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        mae_excess = max(
            0.0, val_metrics["mae"] - baseline_val_mae - MAX_VAL_MAE_INCREASE
        )
        monitor = val_metrics["subject_gender_gap"] + FAIRNESS_MONITOR_MAE_PENALTY * mae_excess
        scheduler.step(monitor)
        history.append({
            "epoch": epoch + 1,
            "train_mae": train_metrics["mae"],
            "train_gap_loss": train_metrics["gap_loss"],
            "val_mae": val_metrics["mae"],
            "val_subject_gap": val_metrics["subject_gender_gap"],
            "selection_monitor": monitor,
        })
        if epoch >= FAIRNESS_WARMUP_EPOCHS and early_stopping.step(monitor, model, epoch):
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_predictions = evaluate_loader(model, test_loader, test_df)
    val_path = os.path.join(REPEATED_SPLIT_RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_path = os.path.join(REPEATED_SPLIT_RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_path)
    ev.save_prediction_frame(test_predictions, test_path)
    return {
        "model": "gender_regularized",
        "split_id": split_id,
        "run_seed": run_seed,
        "fairness_lambda": REPEATED_SPLIT_FAIRNESS_LAMBDA,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "model_path": model_path,
        "val_prediction_path": val_path,
        "test_prediction_path": test_path,
        **{f"val_{key}": value for key, value in val_metrics.items() if not isinstance(value, dict)},
        **{f"test_{key}": value for key, value in test_metrics.items() if not isinstance(value, dict)},
    }


In [ ]:
repeated_split_rows = []
for _, baseline_row in repeated_split_baseline_results.sort_values("split_id").iterrows():
    repeated_split_rows.append(
        train_repeated_split_candidate(
            split_id=int(baseline_row["split_id"]),
            run_seed=int(baseline_row["run_seed"]),
            baseline_val_mae=float(baseline_row["val_mae"]),
        )
    )

repeated_split_results = pd.DataFrame(repeated_split_rows).sort_values("split_id")
comparison = repeated_split_baseline_results.merge(
    repeated_split_results,
    on=["split_id", "run_seed"],
    suffixes=("_baseline", "_candidate"),
    validate="one_to_one",
)
comparison["overall_mae_gain"] = (
    comparison["test_mae_baseline"] - comparison["test_mae_candidate"]
)
comparison["val_overall_mae_gain"] = (
    comparison["val_mae_baseline"] - comparison["val_mae_candidate"]
)
comparison["val_worst_group_mae_gain"] = (
    comparison["val_gender_worst_group_mae_baseline"]
    - comparison["val_gender_worst_group_mae_candidate"]
)
comparison["val_gap_reduction"] = (
    comparison["val_gender_gap_baseline"] - comparison["val_gender_gap_candidate"]
)
comparison["val_subject_worst_group_mae_gain"] = (
    comparison["val_subject_gender_worst_group_mae_baseline"]
    - comparison["val_subject_gender_worst_group_mae_candidate"]
)
comparison["val_subject_gap_reduction"] = (
    comparison["val_subject_gender_gap_baseline"]
    - comparison["val_subject_gender_gap_candidate"]
)
comparison["worst_group_mae_gain"] = (
    comparison["test_gender_worst_group_mae_baseline"]
    - comparison["test_gender_worst_group_mae_candidate"]
)
comparison["gap_reduction"] = (
    comparison["test_gender_gap_baseline"] - comparison["test_gender_gap_candidate"]
)
comparison["subject_worst_group_mae_gain"] = (
    comparison["test_subject_gender_worst_group_mae_baseline"]
    - comparison["test_subject_gender_worst_group_mae_candidate"]
)
comparison["subject_gap_reduction"] = (
    comparison["test_subject_gender_gap_baseline"]
    - comparison["test_subject_gender_gap_candidate"]
)

comparison_metrics = [
    "val_overall_mae_gain", "val_worst_group_mae_gain", "val_gap_reduction",
    "val_subject_worst_group_mae_gain", "val_subject_gap_reduction",
    "overall_mae_gain", "worst_group_mae_gain", "gap_reduction",
    "subject_worst_group_mae_gain", "subject_gap_reduction",
]
summary_rows = []
for metric in comparison_metrics:
    values = comparison[metric]
    summary_rows.append({
        "metric": metric,
        "mean": values.mean(),
        "std": values.std(),
        "median": values.median(),
        "positive_rate": float((values > 0).mean()),
    })
repeated_split_comparison_summary = pd.DataFrame(summary_rows)

repeated_split_results.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "repeated_split_results.csv"), index=False
)
comparison.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "paired_baseline_comparison.csv"), index=False
)
repeated_split_comparison_summary.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "paired_baseline_comparison_summary.csv"),
    index=False,
)
display(comparison[[
    "split_id", "run_seed",
    "val_gap_reduction", "val_subject_gap_reduction",
    "test_mae_baseline", "test_mae_candidate",
    "overall_mae_gain", "gap_reduction", "subject_gap_reduction",
]].round(4))
display(repeated_split_comparison_summary.round(4))
print(f"Saved paired repeated-split results to: {REPEATED_SPLIT_RESULTS_DIR}")
